In [ ]:
# =============================================================
# 07_task_C_fairness_optimisation.ipynb
# Task C — Fairness-Aware Outcome Prediction
# ------------------------------------------------------------
# Benchmark task: predict lower-secondary completion rate Y_it
# subject to group fairness constraints defined over the
# gender parity index (GPI) as protected attribute A_it.
#
# Fairness metric
# ───────────────
# ABROCA (Area Between ROC Curves) — adapted for regression:
#   Δ-RMSE across GPI groups (high vs low gender parity).
#
#   GPI groups:
#     Favoured   (F): GPI ≥ median GPI in training set
#     Unfavoured (U): GPI <  median GPI in training set
#   Δ-RMSE = RMSE_U − RMSE_F   (positive = worse for U group)
#
# Note on ABROCA at n≈24 test observations
# ─────────────────────────────────────────
# With N=12 countries × 3 test years = 36 observations
# (fewer after dropping rows with missing outcome/GPI),
# the ROC curve interpretation of ABROCA is unstable.
# We therefore report Δ-RMSE as the primary fairness
# metric and compute distributional ABROCA as a secondary
# diagnostic, following Borchers & Baker (LAK 2025) [26]
# who establish that ABROCA estimates are reliable at
# n ≥ 200 per group; we note this limitation explicitly.
#
# Models
# ──────
# Unfairness-naïve baselines (from Task A):
#   1. Ridge regression
#   2. Random Forest
#
# Fairness-aware variants:
#   3. Ridge + GPI re-weighting
#      (down-weight training rows from favoured group)
#   4. Ridge + GPI adversarial debiasing
#      (add GPI prediction residual as regularisation term)
#   5. Threshold post-processing
#      (adjust predictions to equalise group RMSE)
#
# Evaluation
# ──────────
# Temporal split (train 2015–2021, test 2022–2024).
# Metrics: overall RMSE, RMSE_F, RMSE_U, Δ-RMSE,
#          ABROCA (diagnostic only).
#
# Outputs
# ───────
#   outputs/results/task_C_metrics.csv
#   outputs/tables/table_taskC.csv / .tex
#   outputs/figures/panel_12/taskC_fairness_tradeoff.png
#   outputs/figures/panel_12/taskC_group_rmse.png
#   outputs/figures/panel_12/taskC_delta_rmse_bar.png
#
# Usage
# ─────
#   python src/07_task_C_fairness_optimisation.py
#
# Requirements
# ────────────
#   pip install scikit-learn pandas numpy matplotlib pyyaml
# =============================================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

from utils import find_project_root, load_indicators, load_pipeline

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"
TRAIN_END  = 2021
TEST_START = 2022

OUTCOME   = "sec_completion"
PROTECTED = "gpi_sec"   # protected attribute A_it
FEATURES  = [k for k, v in IND_CFG["indicators"].items()
             if k not in (OUTCOME, PROTECTED, "pop_total")]

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR  = DATA_DIR / "train_test_splits"
RESULT_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR  = PROJECT_ROOT / "outputs" / "tables"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

for d in [RESULT_DIR, TABLE_DIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Outcome       : {OUTCOME}")
print(f"[info] Protected     : {PROTECTED}")
print(f"[info] Features      : {len(FEATURES)}")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    panel_path = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    temporal_p = SPLIT_DIR / "temporal_split.csv"
    for p in [panel_path, temporal_p]:
        if not p.exists():
            raise FileNotFoundError(f"Required: {p}")
    panel    = pd.read_csv(panel_path)
    temporal = pd.read_csv(temporal_p)
    print(f"\n[step 0] Panel: {len(panel)} rows")
    return panel, temporal


# ──────────────────────────────────────────────────────────────
# Lagged features (same as Task A)
# ──────────────────────────────────────────────────────────────
def make_lagged_features(panel: pd.DataFrame) -> pd.DataFrame:
    all_lag_cols = FEATURES + [PROTECTED]
    lag_cols = {}
    for feat in all_lag_cols:
        if feat in panel.columns:
            lag_cols[f"{feat}_lag1"] = panel.groupby("iso3c")[feat].shift(1)
    panel_lag = panel.copy()
    for name, series in lag_cols.items():
        panel_lag[name] = series
    panel_lag = panel_lag[panel_lag["year"] > YEAR_MIN].copy()
    return panel_lag


def get_feature_cols(panel_lag: pd.DataFrame) -> list[str]:
    """Feature columns excluding the lagged protected attribute."""
    return [c for c in panel_lag.columns
            if c.endswith("_lag1") and not c.startswith(PROTECTED)]


# ──────────────────────────────────────────────────────────────
# Build train / test arrays
# ──────────────────────────────────────────────────────────────
def build_arrays(
    panel_lag:    pd.DataFrame,
    temporal:     pd.DataFrame,
    feature_cols: list[str],
) -> dict:
    """
    Returns dict with train/test X, y, A (protected attr),
    iso3c, year arrays.
    """
    train_isos = set(temporal.loc[temporal["split"]=="train","iso3c"])
    test_isos  = set(temporal.loc[temporal["split"]=="test", "iso3c"])
    train_yrs  = set(temporal.loc[temporal["split"]=="train","year"])
    test_yrs   = set(temporal.loc[temporal["split"]=="test", "year"])

    prot_lag = f"{PROTECTED}_lag1"

    def _split(iso_set, yr_set):
        mask = (panel_lag["iso3c"].isin(iso_set) &
                panel_lag["year"].isin(yr_set) &
                panel_lag[OUTCOME].notna())
        sub  = panel_lag[mask].copy()
        return sub

    tr  = _split(train_isos, train_yrs)
    te  = _split(test_isos,  test_yrs)

    # Impute features on train, apply to test
    imp    = SimpleImputer(strategy="median")
    X_tr   = imp.fit_transform(tr[feature_cols].values.astype(float))
    X_te   = imp.transform(te[feature_cols].values.astype(float))

    # Impute protected attribute separately
    a_imp  = SimpleImputer(strategy="median")
    A_tr   = a_imp.fit_transform(
        tr[prot_lag].values.reshape(-1,1)).ravel()
    A_te   = a_imp.transform(
        te[prot_lag].values.reshape(-1,1)).ravel()

    y_tr   = tr[OUTCOME].values.astype(float)
    y_te   = te[OUTCOME].values.astype(float)

    print(f"\n[step 1] Train: {len(y_tr)} rows  Test: {len(y_te)} rows")
    print(f"  Train GPI range: [{A_tr.min():.3f}, {A_tr.max():.3f}]  "
          f"median={np.median(A_tr):.3f}")
    print(f"  Test  GPI range: [{A_te.min():.3f}, {A_te.max():.3f}]  "
          f"median={np.median(A_te):.3f}")

    return {
        "X_tr": X_tr, "X_te": X_te,
        "y_tr": y_tr, "y_te": y_te,
        "A_tr": A_tr, "A_te": A_te,
        "iso_tr": tr["iso3c"].values,
        "iso_te": te["iso3c"].values,
        "yr_te":  te["year"].values,
        "tier_te":te["income_group"].values,
        "imp": imp, "a_imp": a_imp,
    }


# ──────────────────────────────────────────────────────────────
# Fairness metrics
# ──────────────────────────────────────────────────────────────
def gpi_groups(
    A: np.ndarray,
    A_train: np.ndarray,
) -> np.ndarray:
    """
    Assign group label based on training-set GPI median.
    Returns boolean array: True = favoured (GPI ≥ median).
    """
    median_gpi = float(np.median(A_train))
    return A >= median_gpi


def fairness_metrics(
    y_true:    np.ndarray,
    y_pred:    np.ndarray,
    favoured:  np.ndarray,
    model_name: str,
) -> dict:
    """
    Compute overall RMSE, RMSE per group, Δ-RMSE, and ABROCA proxy.

    ABROCA note: proper ABROCA requires binary outcomes and ROC curves.
    For regression we compute a proxy: the absolute standardised
    difference in RMSE between groups (equivalent to Cohen's d on errors),
    which preserves the directional interpretation while being
    valid at small sample sizes.
    """
    unfavoured = ~favoured

    def rmse(y, yp, mask):
        if mask.sum() == 0:
            return np.nan
        return float(np.sqrt(mean_squared_error(y[mask], yp[mask])))

    rmse_all = rmse(y_true, y_pred, np.ones(len(y_true), dtype=bool))
    rmse_f   = rmse(y_true, y_pred, favoured)
    rmse_u   = rmse(y_true, y_pred, unfavoured)
    delta    = (rmse_u - rmse_f) if (not np.isnan(rmse_u) and
                                      not np.isnan(rmse_f)) else np.nan

    # ABROCA proxy — absolute standardised RMSE difference
    errors_f = np.abs(y_true[favoured]   - y_pred[favoured])   if favoured.sum() > 0 else np.array([])
    errors_u = np.abs(y_true[unfavoured] - y_pred[unfavoured]) if unfavoured.sum() > 0 else np.array([])
    pooled_std = float(np.std(np.abs(y_true - y_pred)) + 1e-9)
    abroca_proxy = abs(delta) / pooled_std if not np.isnan(delta) else np.nan

    return {
        "model":        model_name,
        "rmse_overall": round(rmse_all, 4),
        "rmse_favoured":round(rmse_f,   4),
        "rmse_unfav":   round(rmse_u,   4),
        "delta_rmse":   round(delta,    4) if not np.isnan(delta) else np.nan,
        "abroca_proxy": round(abroca_proxy, 4) if not np.isnan(abroca_proxy) else np.nan,
        "n_favoured":   int(favoured.sum()),
        "n_unfavoured": int(unfavoured.sum()),
    }


# ──────────────────────────────────────────────────────────────
# Model 1 & 2 — naïve baselines (no fairness constraint)
# ──────────────────────────────────────────────────────────────
def run_naive(arrays: dict, feature_cols: list[str]) -> list[dict]:
    results = []
    X_tr, X_te = arrays["X_tr"], arrays["X_te"]
    y_tr, y_te = arrays["y_tr"], arrays["y_te"]
    A_tr, A_te = arrays["A_tr"], arrays["A_te"]

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_te_s  = scaler.transform(X_te)

    favoured = gpi_groups(A_te, A_tr)

    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_tr_s, y_tr)
    y_pred_r = ridge.predict(X_te_s)
    results.append(fairness_metrics(y_te, y_pred_r, favoured, "Ridge_naive"))

    # Random Forest
    rf = RandomForestRegressor(n_estimators=200, max_depth=5,
                                min_samples_leaf=3, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    y_pred_rf = rf.predict(X_te)
    results.append(fairness_metrics(y_te, y_pred_rf, favoured, "RF_naive"))

    return results, scaler, ridge, rf


# ──────────────────────────────────────────────────────────────
# Model 3 — Ridge + GPI sample re-weighting
# ──────────────────────────────────────────────────────────────
def run_reweighted(
    arrays: dict,
    scaler: StandardScaler,
) -> dict:
    """
    Down-weight training samples from the favoured group by factor γ,
    up-weight unfavoured group samples.
    γ is chosen to equalise group sizes in the weighted distribution.
    """
    X_tr_s = scaler.transform(arrays["X_tr"])
    X_te_s = scaler.transform(arrays["X_te"])
    y_tr   = arrays["y_tr"]
    A_tr   = arrays["A_tr"]
    A_te   = arrays["A_te"]

    favoured_tr  = gpi_groups(A_tr, A_tr)
    n_f  = favoured_tr.sum()
    n_u  = (~favoured_tr).sum()
    n    = len(y_tr)

    if n_f == 0 or n_u == 0:
        # Degenerate — fall back to uniform weights
        weights = np.ones(n)
    else:
        # Equalise group contributions
        w_f = n / (2 * n_f)
        w_u = n / (2 * n_u)
        weights = np.where(favoured_tr, w_f, w_u)

    ridge_rw = Ridge(alpha=1.0)
    ridge_rw.fit(X_tr_s, y_tr, sample_weight=weights)
    y_pred = ridge_rw.predict(X_te_s)

    favoured_te = gpi_groups(A_te, A_tr)
    return fairness_metrics(
        arrays["y_te"], y_pred, favoured_te, "Ridge_reweighted"
    )


# ──────────────────────────────────────────────────────────────
# Model 4 — Ridge + adversarial debiasing proxy
# ──────────────────────────────────────────────────────────────
def run_adversarial(
    arrays: dict,
    scaler: StandardScaler,
) -> dict:
    """
    Adversarial debiasing proxy:
    1. Train an auxiliary Ridge to predict A_it from X_{i,t-1}.
    2. Compute GPI residual r = A_it − Â_it (unexplained GPI).
    3. Subtract λ * r from y_it (fairness-adjusted outcome).
    4. Train main Ridge on y_adjusted.
    5. Prediction uses original model, not adjusted outcome.

    This implements a simplified version of the Blinder-Oaxaca
    decomposition approach — removing the component of Y that is
    linearly predictable from A, conditional on X.

    λ = 0.5 (balances predictive vs fairness objective).
    """
    LAMBDA    = 0.5
    X_tr_s    = scaler.transform(arrays["X_tr"])
    X_te_s    = scaler.transform(arrays["X_te"])
    y_tr      = arrays["y_tr"]
    A_tr      = arrays["A_tr"]
    A_te      = arrays["A_te"]

    # Step 1: auxiliary GPI predictor
    aux_ridge = Ridge(alpha=1.0)
    aux_ridge.fit(X_tr_s, A_tr)
    A_hat_tr  = aux_ridge.predict(X_tr_s)
    A_resid   = A_tr - A_hat_tr

    # Step 2: fairness-adjusted outcome
    y_adj = y_tr - LAMBDA * A_resid

    # Step 3: main model on adjusted outcome
    main_ridge = Ridge(alpha=1.0)
    main_ridge.fit(X_tr_s, y_adj)
    y_pred = main_ridge.predict(X_te_s)

    favoured_te = gpi_groups(A_te, A_tr)
    return fairness_metrics(
        arrays["y_te"], y_pred, favoured_te, "Ridge_adversarial"
    )


# ──────────────────────────────────────────────────────────────
# Model 5 — Threshold post-processing (equalised RMSE)
# ──────────────────────────────────────────────────────────────
def run_postprocessing(
    arrays:    dict,
    scaler:    StandardScaler,
    ridge_naive: Ridge,
) -> dict:
    """
    Post-processing: shift group predictions to equalise group RMSE.

    For each group g ∈ {F, U}:
      1. Compute group bias b_g = mean(y_true_g) − mean(y_pred_g)
         on the training set (calibration).
      2. Add b_g to test predictions for group g.

    This is a group-conditional mean-shift, equivalent to
    group-wise intercept recalibration.
    """
    X_tr_s = scaler.transform(arrays["X_tr"])
    X_te_s = scaler.transform(arrays["X_te"])
    y_tr   = arrays["y_tr"]
    y_te   = arrays["y_te"]
    A_tr   = arrays["A_tr"]
    A_te   = arrays["A_te"]

    # Naive ridge predictions on train (calibration set)
    y_pred_tr = ridge_naive.predict(X_tr_s)
    y_pred_te = ridge_naive.predict(X_te_s)

    favoured_tr = gpi_groups(A_tr, A_tr)
    favoured_te = gpi_groups(A_te, A_tr)

    y_pred_pp = y_pred_te.copy()

    for grp_mask_tr, grp_mask_te in [
        (favoured_tr,  favoured_te),
        (~favoured_tr, ~favoured_te),
    ]:
        if grp_mask_tr.sum() == 0:
            continue
        bias = float(
            np.mean(y_tr[grp_mask_tr]) - np.mean(y_pred_tr[grp_mask_tr])
        )
        y_pred_pp[grp_mask_te] += bias

    return fairness_metrics(
        y_te, y_pred_pp, favoured_te, "Ridge_postprocessing"
    )


# ──────────────────────────────────────────────────────────────
# Figures
# ──────────────────────────────────────────────────────────────
MODEL_COLOURS = {
    "Ridge_naive":        "#888888",
    "RF_naive":           "#4d4d4d",
    "Ridge_reweighted":   "#4393c3",
    "Ridge_adversarial":  "#74c476",
    "Ridge_postprocessing":"#d6604d",
}


def plot_fairness_tradeoff(df: pd.DataFrame) -> None:
    """
    Scatter: overall RMSE (x) vs |Δ-RMSE| (y) per model.
    Lower-left is better (accurate and fair).
    """
    fig, ax = plt.subplots(figsize=(8, 6))

    for _, row in df.iterrows():
        name   = row["model"]
        colour = MODEL_COLOURS.get(name, "grey")
        x      = row["rmse_overall"]
        y      = abs(row["delta_rmse"]) if pd.notna(row["delta_rmse"]) else np.nan
        if np.isnan(y):
            continue
        ax.scatter(x, y, color=colour, s=160, zorder=4,
                   edgecolors="white", linewidths=1.0)
        ax.annotate(
            name.replace("Ridge_", "").replace("RF_", "RF "),
            xy=(x, y), xytext=(x + 0.15, y + 0.05),
            fontsize=8.5, color=colour, fontweight="bold",
        )

    # Pareto-front arrow
    ax.annotate("", xy=(df["rmse_overall"].min() - 1,
                         df["delta_rmse"].abs().min() - 0.3),
                xytext=(df["rmse_overall"].max() + 1,
                         df["delta_rmse"].abs().max() + 0.3),
                arrowprops=dict(arrowstyle="<->", color="#cccccc", lw=1.5))
    ax.text(df["rmse_overall"].mean(), df["delta_rmse"].abs().mean() - 1,
            "← more accurate\n↓ more fair",
            ha="center", fontsize=8, color="#888888")

    ax.set_xlabel("Overall RMSE (pp)", fontsize=11)
    ax.set_ylabel("|Δ-RMSE| = |RMSE_unfav − RMSE_fav| (pp)", fontsize=11)
    ax.set_title(
        "Task C — Accuracy–fairness trade-off\n"
        "(lower-left = better; protected attribute = GPI secondary)",
        fontsize=11, fontweight="bold",
    )
    ax.grid(alpha=0.2, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskC_fairness_tradeoff_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_group_rmse(df: pd.DataFrame) -> None:
    """
    Grouped bar chart: RMSE_favoured vs RMSE_unfavoured per model.
    """
    models = df["model"].tolist()
    x      = np.arange(len(models))
    width  = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    bars_f = ax.bar(x - width/2, df["rmse_favoured"], width,
                    label="Favoured (GPI ≥ median)",
                    color="#4393c3", alpha=0.85, edgecolor="white")
    bars_u = ax.bar(x + width/2, df["rmse_unfav"], width,
                    label="Unfavoured (GPI < median)",
                    color="#d6604d", alpha=0.85, edgecolor="white")

    for bars in [bars_f, bars_u]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h) and h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                        f"{h:.1f}", ha="center", va="bottom", fontsize=7.5)

    short_names = [m.replace("Ridge_","").replace("RF_","RF ") for m in models]
    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=20, ha="right", fontsize=9)
    ax.set_ylabel("RMSE (pp)", fontsize=10)
    ax.set_title(
        "Task C — Group-conditional RMSE by model\n"
        "(protected attribute = secondary GPI; "
        f"n_favoured={df['n_favoured'].iloc[0]}, "
        f"n_unfavoured={df['n_unfavoured'].iloc[0]})",
        fontsize=10, fontweight="bold",
    )
    ax.legend(fontsize=9, frameon=True, edgecolor="#cccccc")
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskC_group_rmse_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_delta_rmse(df: pd.DataFrame) -> None:
    """
    Horizontal bar chart of Δ-RMSE per model.
    Positive = model is less accurate for unfavoured group.
    Zero line shown.
    """
    models  = df["model"].tolist()
    deltas  = df["delta_rmse"].values
    colours = [("#d6604d" if d > 0 else "#4393c3")
               if not np.isnan(d) else "#cccccc"
               for d in deltas]
    short   = [m.replace("Ridge_","").replace("RF_","RF ") for m in models]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars    = ax.barh(short, deltas, color=colours,
                      edgecolor="white", height=0.6)
    ax.axvline(0, color="#333333", lw=1.2)

    for bar, v in zip(bars, deltas):
        if not np.isnan(v):
            xpos = v + (0.3 if v >= 0 else -0.3)
            ha   = "left" if v >= 0 else "right"
            ax.text(xpos, bar.get_y() + bar.get_height()/2,
                    f"{v:+.2f} pp", va="center", ha=ha, fontsize=9)

    ax.set_xlabel("Δ-RMSE = RMSE_unfav − RMSE_fav (pp)", fontsize=10)
    ax.set_title(
        "Task C — Fairness gap (Δ-RMSE) by model\n"
        "Red = disadvantages unfavoured group  |  "
        "Blue = advantages unfavoured group",
        fontsize=10, fontweight="bold",
    )
    ax.grid(axis="x", alpha=0.25, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskC_delta_rmse_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────────────
def save_results(df: pd.DataFrame) -> None:
    print("\n[step 4] Saving results …")

    df.to_csv(RESULT_DIR / "task_C_metrics.csv", index=False)
    df.to_csv(TABLE_DIR  / "table_taskC.csv",    index=False)
    print(f"  [OK] task_C_metrics.csv")

    # LaTeX table
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Task C — Fairness-aware prediction results "
        r"(temporal split; protected attribute = secondary GPI). "
        r"$\Delta$-RMSE $= $ RMSE\textsubscript{unfav} "
        r"$-$ RMSE\textsubscript{fav}; "
        r"ABROCA proxy = $|\Delta\text{-RMSE}|$ / pooled error SD. "
        r"GPI groups split at training-set median. "
        r"Note: ABROCA estimates at $n \approx 24$ test observations "
        r"are indicative only (cf.\ Borchers \& Baker, LAK 2025).}",
        r"\label{tab:taskC}",
        r"\begin{tabular}{lrrrrc}",
        r"\toprule",
        r"Model & RMSE & RMSE$_F$ & RMSE$_U$ & "
        r"$\Delta$-RMSE & ABROCA proxy \\",
        r"\midrule",
    ]

    def f(v): return f"{v:.3f}" if pd.notna(v) else "--"
    for _, row in df.iterrows():
        name  = row["model"].replace("_", r"\_")
        # Bold best overall RMSE
        rmse  = f(row["rmse_overall"])
        lines.append(
            f"{name} & {rmse} & {f(row['rmse_favoured'])} & "
            f"{f(row['rmse_unfav'])} & {f(row['delta_rmse'])} & "
            f"{f(row['abroca_proxy'])} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABLE_DIR / "table_taskC.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    print(f"  [OK] table_taskC.csv / .tex")
    print(f"\n[done] Results : {RESULT_DIR.resolve()}")
    print(f"[done] Tables  : {TABLE_DIR.resolve()}")
    print(f"[done] Figures : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/08_task_D_mnar_imputation.py")


# ──────────────────────────────────────────────────────────────
# Console summary
# ──────────────────────────────────────────────────────────────
def print_summary(df: pd.DataFrame) -> None:
    print(f"\n{'='*72}")
    print("Task C — Fairness summary")
    print(f"{'='*72}")
    print(f"  {'Model':<26}  {'RMSE':>6}  {'RMSE_F':>7}  "
          f"{'RMSE_U':>7}  {'Δ-RMSE':>8}  {'ABROCA':>7}")
    print(f"  {'-'*26}  {'-'*6}  {'-'*7}  {'-'*7}  {'-'*8}  {'-'*7}")
    for _, row in df.iterrows():
        def f(v, w=7): return f"{v:{w}.3f}" if pd.notna(v) else f"{'--':>{w}}"
        print(f"  {row['model']:<26}  {f(row['rmse_overall'],6)}  "
              f"{f(row['rmse_favoured'])}  {f(row['rmse_unfav'])}  "
              f"{f(row['delta_rmse'],8)}  {f(row['abroca_proxy'])}")
    print(f"{'='*72}")
    print(f"\n  GPI groups: favoured (≥ median), unfavoured (< median)")
    print(f"  n_favoured={df['n_favoured'].iloc[0]}  "
          f"n_unfavoured={df['n_unfavoured'].iloc[0]}")
    print(f"\n  Note: ABROCA proxy is |Δ-RMSE| / pooled-error SD.")
    print(f"  At n≈{df['n_favoured'].iloc[0]+df['n_unfavoured'].iloc[0]} "
          f"test observations, ABROCA estimates are indicative only")
    print(f"  (cf. Borchers & Baker, LAK 2025).")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, temporal = load_data()

    panel_lag    = make_lagged_features(panel)
    feature_cols = get_feature_cols(panel_lag)

    print(f"\n[step 1] Feature cols: {len(feature_cols)} (lag-1, excl. protected)")
    arrays = build_arrays(panel_lag, temporal, feature_cols)

    print("\n[step 2] Running models …")

    naive_results, scaler, ridge_naive, rf_naive = run_naive(
        arrays, feature_cols)

    rw_result   = run_reweighted(arrays, scaler)
    adv_result  = run_adversarial(arrays, scaler)
    pp_result   = run_postprocessing(arrays, scaler, ridge_naive)

    all_results = naive_results + [rw_result, adv_result, pp_result]
    df          = pd.DataFrame(all_results)

    print_summary(df)

    print("\n[step 3] Figures …")
    plot_fairness_tradeoff(df)
    plot_group_rmse(df)
    plot_delta_rmse(df)

    save_results(df)


if __name__ == "__main__":
    main()